# 01 — CWPP alerts overview

Bird's-eye view of every Microsoft Defender for Cloud alert in the workspace.
Use it for daily standup, demo wrap-up, or as a starting point before drilling
into a specific incident.

**You will need**

- `LAW_WORKSPACE_ID` set to the *customer ID* (GUID) of your Log Analytics workspace — find it in
  Portal → Log Analytics → Properties → Workspace ID.
- `pip install azure-identity azure-monitor-query pandas matplotlib`
- `az login` (or any other credential picked up by `DefaultAzureCredential`).


In [ ]:
"""Shared setup — run me first.
Connects to Log Analytics via DefaultAzureCredential (uses az login, env vars, or managed identity).
"""
import os
import pandas as pd
from datetime import timedelta
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

WORKSPACE_ID = os.environ.get("LAW_WORKSPACE_ID")  # GUID of the Log Analytics workspace
assert WORKSPACE_ID, "Set LAW_WORKSPACE_ID env var (workspace customerId GUID, not full ARM id)"

client = LogsQueryClient(DefaultAzureCredential())

def kql(query: str, hours: int = 24) -> pd.DataFrame:
    """Run a KQL query against the workspace and return the first table as a DataFrame."""
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        print("WARNING: partial result -", resp.partial_error)
        tables = resp.partial_data
    elif resp.status == LogsQueryStatus.SUCCESS:
        tables = resp.tables
    else:
        raise RuntimeError(f"Query failed: {resp}")
    if not tables:
        return pd.DataFrame()
    t = tables[0]
    return pd.DataFrame(t.rows, columns=t.columns)

pd.set_option("display.max_colwidth", 120)
print("Workspace:", WORKSPACE_ID)


## Total alerts by severity (last 24h)

In [ ]:
df = kql("""
SecurityAlert
| where TimeGenerated > ago(24h)
| where ProviderName == "Azure Security Center"
| summarize Count=count() by AlertSeverity
| order by Count desc
""")
df


## Alerts by plan / resource type

In [ ]:
df = kql("""
SecurityAlert
| where TimeGenerated > ago(24h)
| where ProviderName == "Azure Security Center"
| extend Plan = tostring(parse_json(ExtendedProperties).["Defender plan"])
| extend ResourceType = tostring(parse_json(ExtendedProperties).["Resource type"])
| summarize Count=count() by Plan, ResourceType
| order by Count desc
""")
df


## Alert volume over time

In [ ]:
import matplotlib.pyplot as plt
df = kql("""
SecurityAlert
| where TimeGenerated > ago(24h)
| where ProviderName == "Azure Security Center"
| summarize Count=count() by bin(TimeGenerated, 30m), AlertSeverity
""")
pivot = df.pivot(index="TimeGenerated", columns="AlertSeverity", values="Count").fillna(0)
ax = pivot.plot(kind="bar", stacked=True, figsize=(12,4), title="CWPP alerts (30-min bins)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


## MITRE tactic coverage observed

In [ ]:
df = kql("""
SecurityAlert
| where TimeGenerated > ago(7d)
| where ProviderName == "Azure Security Center"
| mv-expand Tactics=parse_json(Tactics)
| extend Tactic=tostring(Tactics)
| summarize Count=count() by Tactic
| order by Count desc
""", hours=24*7)
df


## Top compromised entities

In [ ]:
df = kql("""
SecurityAlert
| where TimeGenerated > ago(24h)
| where ProviderName == "Azure Security Center"
| summarize AlertCount=count(), Sample=any(AlertName) by CompromisedEntity, AlertSeverity
| where AlertCount > 0
| order by AlertCount desc
| take 20
""")
df
